## 1.0 Ingest

In [2]:

import requests
from bs4 import BeautifulSoup

url = "https://www.bcg.com/india/latest-economic-trends-in-india"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

pdf_links = set() # Use a set to avoid duplicates

# Find all <a> tags
for link in soup.find_all('a'):
    # Check multiple attributes where the URL might be stored
    # 1. Standard href
    # 2. data-href (used by this site)
    # 3. data-analytics-file-name (also used by this site)
    potential_urls = [
        link.get('href'),
        link.get('data-href'),
        link.get('data-analytics-file-name')
    ]
    
    for val in potential_urls:
        if val and val.lower().endswith('.pdf'):
            # Ensure it's a full URL
            if not val.startswith('http'):
                from urllib.parse import urljoin
                val = urljoin(url, val)
            pdf_links.add(val)

print(f"Found {len(pdf_links)} PDF(s):")
for pdf in pdf_links:
    print(pdf)

Found 73 PDF(s):
https://media-publications.bcg.com/BCG-India-Economic-Monitor-Sep-2020.pdf
https://web-assets.bcg.com/dc/6e/4dee2def4173bb0b7082ee951190/india-economic-monitor-aug25.pdf
https://web-assets.bcg.com/b9/46/cedcaa0f4105bd677137f87e0eb2/india-economic-monitor-oct24.pdf
https://web-assets.bcg.com/75/5d/590425af4aa094bebdf0b2a04012/bcg-india-economic-monitor-april-2023.pdf
https://media-publications.bcg.com/BCG-India-Economic-Monitor-Jul-2022.pdf
https://media-publications.bcg.com/BCG-India-Economic-Monitor-June-2022.pdf
https://web-assets.bcg.com/f1/d9/7a5d048c42e2a8426611855a5af1/india-economic-monitor-dec24.pdf
https://web-assets.bcg.com/21/44/0ab4ce7448c8bf1b995539727c33/bcg-india-economic-monitor-march-2023.pdf
https://media-publications.bcg.com/BCG-India-Economic-Monitor-Dec-2020.pdf
https://web-assets.bcg.com/be/1e/eebd0f6b470692f2cecc3f320718/bcg-india-economic-monitor-february-2023.pdf
https://media-publications.bcg.com/BCG-India-Economic-Monitor-November-2020.pdf
ht

In [3]:
pdf_links = {link for link in pdf_links if link.startswith("https://web-assets")}

In [6]:
len(pdf_links)

44

## 2.0 Multimodal processing, structured facts & hybrid retrieval

Standalone pipeline (no external package deps) over `iem-sep26.pdf`:
**parse → multimodal chart understanding → structured observations → semantic chunks → embeddings → hybrid retrieval**, with provenance on every fact.
Runs fully offline (hash embedder + Null vision provider); set `OPENAI_API_KEY` to enable real chart extraction and `DATABASE_URL` to mirror into pgvector.

In [1]:
# === 2.0 Multimodal pipeline — STANDALONE setup (no external package deps) ===
# Everything this notebook needs is inlined below so it runs on a bare
# `pyenv global 3.13.1` with: pymupdf, numpy, pillow, requests, and (optional) openai.
from __future__ import annotations

import base64
import hashlib
import html
import json
import math
import os
import re
import unicodedata
from collections import Counter
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import pymupdf  # PyMuPDF 2.x — `fitz` alias is deprecated

# --------------------------------------------------------------------------- #
# Text helpers (inlined — previously imported from optrade_data.processing)
# --------------------------------------------------------------------------- #
_WS = re.compile(r"[ \t\f\v]+")
_NL = re.compile(r"\n{3,}")
_TAG = re.compile(r"<[^>]+>")


def clean_text(text: str) -> str:
    text = html.unescape(text)
    text = _TAG.sub(" ", text)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = "\n".join(_WS.sub(" ", line).strip() for line in text.split("\n"))
    return _NL.sub("\n\n", text).strip()


def chunk_text(text: str, size: int = 800, overlap: int = 100) -> list[str]:
    """~`size`-char chunks with `overlap`, preferring whitespace boundaries."""
    if size <= 0 or overlap < 0 or overlap >= size:
        raise ValueError("require size > 0 and 0 <= overlap < size")
    text = text.strip()
    if not text:
        return []
    chunks, start, n = [], 0, len(text)
    while start < n:
        end = min(start + size, n)
        if end < n:
            cut = text.rfind(" ", start + size // 2, end)
            if cut != -1:
                end = cut
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)
        if end >= n:
            break
        start = max(end - overlap, start + 1)
        if start > 0 and text[start - 1] != " ":
            nxt = text.find(" ", start, end)
            if nxt != -1:
                start = nxt + 1
    return chunks


# --------------------------------------------------------------------------- #
# Embedding abstraction (hash default = offline + deterministic; ST optional)
# --------------------------------------------------------------------------- #
def hash_embed(text: str, dim: int = 384) -> list[float]:
    vec = [0.0] * dim
    for tok in re.findall(r"\w+", text.lower()):
        h = int(hashlib.md5(tok.encode()).hexdigest(), 16)
        vec[h % dim] += 1.0 if (h >> 64) & 1 else -1.0
    norm = math.sqrt(sum(v * v for v in vec)) or 1.0
    return [v / norm for v in vec]


def to_pgvector(vec) -> str:
    return "[" + ",".join(f"{v:.6f}" for v in vec) + "]"


class HashEmbedder:
    def __init__(self, dim: int = 384):
        self.dim = dim

    def embed(self, texts: list[str]) -> list[list[float]]:
        return [hash_embed(t, self.dim) for t in texts]


class SentenceTransformerEmbedder:
    def __init__(self, model_name: str, dim: int):
        from sentence_transformers import SentenceTransformer

        self.model = SentenceTransformer(model_name)
        self.dim = dim

    def embed(self, texts: list[str]) -> list[list[float]]:
        vecs = self.model.encode(texts, normalize_embeddings=True)
        return [list(map(float, v)) for v in vecs]


def get_embedder(backend: str, dim: int, model: str | None = None):
    if backend == "sentence-transformers":
        return SentenceTransformerEmbedder(model or "sentence-transformers/all-MiniLM-L6-v2", dim)
    if backend != "hash":
        raise ValueError(f"unknown embedding backend: {backend}")
    return HashEmbedder(dim)


# --------------------------------------------------------------------------- #
# Macro domain dictionaries — indicators, institutions, geographies
# --------------------------------------------------------------------------- #
INDICATORS: dict[str, list[str]] = {
    "CPI": ["cpi", "consumer price index", "retail inflation", "headline inflation"],
    "Core CPI": ["core cpi", "core inflation"],
    "Food Inflation": ["food inflation", "cfpi", "food price index"],
    "WPI": ["wpi", "wholesale price index"],
    "GDP": ["gdp", "gross domestic product"],
    "GVA": ["gva", "gross value added"],
    "IIP": ["iip", "index of industrial production", "industrial production"],
    "PMI": ["pmi", "purchasing managers", "manufacturing pmi", "services pmi"],
    "Repo Rate": ["repo rate", "policy rate", "repo"],
    "CRR": ["crr", "cash reserve ratio"],
    "Credit Growth": ["credit growth", "bank credit", "non-food credit", "loan growth"],
    "Deposit Growth": ["deposit growth", "bank deposits"],
    "Fiscal Deficit": ["fiscal deficit"],
    "Current Account": ["current account deficit", "current account", "cad"],
    "Trade Balance": ["trade balance", "trade deficit", "merchandise trade"],
    "Forex Reserves": ["forex reserves", "foreign exchange reserves", "fx reserves"],
    "Brent Crude": ["brent", "crude oil", "crude", "brent crude"],
    "FDI": ["fdi", "foreign direct investment"],
    "FII": ["fii", "fpi", "foreign portfolio", "foreign institutional"],
    "Unemployment": ["unemployment", "jobless rate"],
    "Rupee": ["rupee", "inr", "usd/inr", "exchange rate"],
    "GST": ["gst", "goods and services tax", "gst collections"],
}
_IND_LOOKUP = {a: canon for canon, al in INDICATORS.items() for a in al}
_IND_RE = re.compile(
    r"\b(" + "|".join(sorted((re.escape(a) for a in _IND_LOOKUP), key=len, reverse=True)) + r")\b",
    re.I,
)

INSTITUTIONS = ["RBI", "Reserve Bank of India", "Government of India", "MoSPI", "SEBI",
                "Federal Reserve", "Fed", "IMF", "World Bank", "OPEC", "NSO", "CSO"]
GEOGRAPHIES = ["India", "US", "USA", "United States", "China", "Europe", "Eurozone",
               "UK", "Japan", "Asia", "Global"]
_ENT_RE = {
    "INSTITUTION": re.compile(r"\b(" + "|".join(map(re.escape, INSTITUTIONS)) + r")\b"),
    "GEO": re.compile(r"\b(" + "|".join(map(re.escape, GEOGRAPHIES)) + r")\b"),
}
_MONEY_RE = re.compile(
    r"(?:₹|\$|Rs\.?)\s?\d[\d,]*(?:\.\d+)?\s?"
    r"(?:lakh\s+crore|crore|lakh|billion|million|trillion|bn|mn|tn)?",
    re.I,
)
_PCT_RE = re.compile(r"-?\d+(?:\.\d+)?\s?(?:%|percent|per cent|bps)", re.I)


@dataclass(frozen=True)
class Entity:
    text: str
    label: str


def extract_entities(text: str) -> list[Entity]:
    out: list[Entity] = []
    seen: set[tuple[str, str]] = set()

    def add(t: str, label: str):
        key = (t.strip(), label)
        if t.strip() and key not in seen:
            seen.add(key)
            out.append(Entity(t.strip(), label))

    for m in _IND_RE.finditer(text):
        add(_IND_LOOKUP[m.group(1).lower()], "INDICATOR")
    for label, rx in _ENT_RE.items():
        for m in rx.finditer(text):
            add(m.group(1), label)
    for m in _MONEY_RE.finditer(text):
        add(m.group(0), "MONEY")
    for m in _PCT_RE.finditer(text):
        add(m.group(0), "PERCENT")
    return out


def normalize_indicator(text: str) -> str | None:
    m = _IND_RE.search(text or "")
    return _IND_LOOKUP[m.group(1).lower()] if m else None


# --------------------------------------------------------------------------- #
# Configuration — all swappable via environment variables
# --------------------------------------------------------------------------- #
@dataclass
class Config:
    embed_backend: str = os.environ.get("EMBEDDING_BACKEND", "hash")
    embed_model: str | None = os.environ.get("EMBEDDING_MODEL")
    embed_dim: int = int(os.environ.get("EMBEDDING_DIM", "384"))
    chunk_size: int = 800
    chunk_overlap: int = 100
    render_dpi: int = 150
    # multimodal chart understanding
    mm_provider: str = os.environ.get("MULTIMODAL_PROVIDER", "openai")
    mm_model: str = os.environ.get("MULTIMODAL_MODEL", "gpt-4o-mini")
    mm_max_pages: int = int(os.environ.get("MULTIMODAL_MAX_PAGES", "0"))  # 0 = no cap
    mm_min_drawings: int = 30  # page is a chart candidate if it has >= this many vector ops
    review_confidence: float = 0.75  # below this → human-review queue


CFG = Config()

STORE = Path.cwd() / "vector_store"
PDF_DIR = Path.cwd() / "pdfs"
FIG_DIR = STORE / "figures"
PAGE_IMG_DIR = STORE / "page_images"
for _d in (STORE, PDF_DIR, FIG_DIR, PAGE_IMG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

EMBEDDER = get_embedder(CFG.embed_backend, CFG.embed_dim, CFG.embed_model)


# --------------------------------------------------------------------------- #
# MultimodalModelProvider — vision LLM abstraction (OpenAI + offline Null)
# --------------------------------------------------------------------------- #
_CHART_SCHEMA_HINT = (
    "Return STRICT JSON: {\"charts\":[{"
    "\"title\":str|null,\"subtitle\":str|null,\"chart_type\":str|null,"
    "\"x_axis\":{\"label\":str|null,\"values\":[str]},"
    "\"y_axis\":{\"label\":str|null,\"unit\":str|null},"
    "\"series\":[{\"name\":str|null,\"values\":[{\"period\":str,\"value\":number}]}],"
    "\"legend\":[str],\"source\":str|null,\"period\":str|null,"
    "\"interpretation\":str,\"confidence\":number}]}. "
    "RULES: Only report values you can read from the image. If a value is not "
    "legible, OMIT it — never guess. If the page has no chart, return {\"charts\":[]}. "
    "confidence in [0,1] reflects how reliably you read the numbers."
)


class NullMultimodalProvider:
    """Offline fallback: never fabricates data; marks charts as unavailable."""

    name = "null"
    available = False

    def describe_page(self, image_path: str, context: str) -> dict:
        return {"charts": [], "_status": "unavailable_offline"}


class OpenAIMultimodalProvider:
    """OpenAI vision provider. Requires OPENAI_API_KEY; model from CFG.mm_model."""

    name = "openai"
    available = True

    def __init__(self, model: str):
        from openai import OpenAI

        self.client = OpenAI()
        self.model = model

    def describe_page(self, image_path: str, context: str) -> dict:
        b64 = base64.b64encode(Path(image_path).read_bytes()).decode()
        resp = self.client.chat.completions.create(
            model=self.model,
            response_format={"type": "json_object"},
            temperature=0,
            messages=[
                {"role": "system",
                 "content": "You extract chart data from economic report pages. " + _CHART_SCHEMA_HINT},
                {"role": "user", "content": [
                    {"type": "text",
                     "text": f"Page text context (for axis/units/source):\n{context[:1500]}"},
                    {"type": "image_url",
                     "image_url": {"url": f"data:image/png;base64,{b64}"}},
                ]},
            ],
        )
        try:
            return json.loads(resp.choices[0].message.content or "{}")
        except json.JSONDecodeError:
            return {"charts": [], "_status": "parse_error"}


def get_multimodal_provider(cfg: Config):
    if cfg.mm_provider == "openai" and os.environ.get("OPENAI_API_KEY"):
        try:
            return OpenAIMultimodalProvider(cfg.mm_model)
        except Exception as exc:  # SDK missing / auth → degrade gracefully
            print("  multimodal: OpenAI unavailable →", exc)
    return NullMultimodalProvider()


MM_PROVIDER = get_multimodal_provider(CFG)


# --------------------------------------------------------------------------- #
# Stage 0 — acquire & register documents (idempotent via content hash)
# --------------------------------------------------------------------------- #
@dataclass
class Document:
    doc_id: str
    source: str
    path: str
    title: str
    edition: str | None
    publication_date: str | None  # ISO month of the report edition
    sha256: str
    n_pages: int


_EDITION = re.compile(
    r"(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\W{0,3}'?\s?(\d{2,4})", re.I
)
_MONTHS = {m: i for i, m in enumerate(
    ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"], 1)}


def _sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def _edition_to_iso(edition: str | None) -> str | None:
    if not edition:
        return None
    mon, yr = edition.split()
    year = int(yr) if len(yr) == 4 else 2000 + int(yr)
    return f"{year:04d}-{_MONTHS.get(mon[:3].lower(), 1):02d}"


def _download(url: str) -> Path | None:
    import requests

    name = url.split("/")[-1].split("?")[0] or "download.pdf"
    if not name.lower().endswith(".pdf"):
        name += ".pdf"
    dest = PDF_DIR / name
    if dest.exists():
        return dest
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        dest.write_bytes(resp.content)
        return dest
    except Exception as exc:
        print("  download failed:", url, exc)
        return None


def register_documents(local_pdfs, urls) -> list[Document]:
    paths: list[tuple[str, Path]] = []
    for lp in local_pdfs:
        p = Path(lp)
        if p.exists():
            paths.append(("local", p))
    for u in sorted(urls or []):
        p = _download(u)
        if p:
            paths.append((u, p))

    docs, seen = [], set()
    for source, p in paths:
        digest = _sha256(p)
        if digest in seen:  # idempotency: skip identical PDFs
            continue
        seen.add(digest)
        pdf = pymupdf.open(p)
        first = pdf.load_page(0).get_text("text") if pdf.page_count else ""
        m = _EDITION.search(first) or _EDITION.search(p.stem)
        edition = f"{m.group(1).title()} {m.group(2)}" if m else None
        title = (pdf.metadata or {}).get("title") or p.stem
        docs.append(Document(digest[:12], source, str(p), title, edition,
                             _edition_to_iso(edition), digest, pdf.page_count))
        pdf.close()
    return docs


_urls = globals().get("pdf_links", set())  # reuse section 1.0 crawl output if present
DOCUMENTS = register_documents([Path.cwd() / "iem-sep26.pdf"], _urls)
print(f"embedder={CFG.embed_backend}({CFG.embed_dim})  multimodal={MM_PROVIDER.name}"
      f"(available={MM_PROVIDER.available})")
for _doc in DOCUMENTS:
    print(f"{_doc.doc_id} · {_doc.edition} · pub={_doc.publication_date} · "
          f"{_doc.n_pages}pp · {_doc.title[:44]} · {_doc.source[:36]}")


embedder=hash(384)  multimodal=null(available=False)
3c287c9e7e87 · Sep 2026 · pub=2026-09 · 28pp · iem-sep26 · local


In [2]:
# === Stage 1: parse & layout (text blocks + tables + figures + page renders) ===
@dataclass
class Element:
    doc_id: str
    page: int
    kind: str  # 'text' | 'table' | 'figure'
    bbox: tuple
    text: str = ""
    font_max: float = 0.0
    ref: str = ""  # saved PNG path for figures


@dataclass
class Page:
    doc_id: str
    page: int
    raw_text: str
    render_path: str
    n_drawings: int
    n_images: int
    is_chart_candidate: bool


def _text_blocks(page):
    out = []
    for b in page.get_text("dict").get("blocks", []):
        if b.get("type") != 0:  # text blocks only
            continue
        parts, fmax = [], 0.0
        for line in b.get("lines", []):
            for span in line.get("spans", []):
                parts.append(span.get("text", ""))
                fmax = max(fmax, float(span.get("size", 0.0)))
        t = clean_text(" ".join(parts))
        if t:
            out.append((tuple(b["bbox"]), t, fmax))
    return out


def parse_document(doc: Document) -> tuple[list[Element], list[Page]]:
    els: list[Element] = []
    pages: list[Page] = []
    pdf = pymupdf.open(doc.path)
    for i in range(pdf.page_count):
        page = pdf.load_page(i)
        blocks = _text_blocks(page)
        for bbox, t, fmax in blocks:
            els.append(Element(doc.doc_id, i, "text", bbox, text=t, font_max=fmax))
        try:
            for tbl in page.find_tables().tables:
                rows = tbl.extract()
                if rows:
                    md = "\n".join(
                        " | ".join("" if c is None else str(c) for c in r) for r in rows
                    )
                    els.append(Element(doc.doc_id, i, "table", tuple(tbl.bbox), text=md))
        except Exception:
            pass
        n_images = 0
        for j, info in enumerate(page.get_images(full=True)):
            xref = info[0]
            try:
                pix = pymupdf.Pixmap(pdf, xref)
                if pix.n > 4:
                    pix = pymupdf.Pixmap(pymupdf.csRGB, pix)
                fp = FIG_DIR / f"{doc.doc_id}_p{i}_img{j}.png"
                pix.save(fp)
                rects = page.get_image_rects(xref)
                bbox = tuple(rects[0]) if rects else (0, 0, 0, 0)
                els.append(Element(doc.doc_id, i, "figure", bbox, ref=str(fp)))
                n_images += 1
            except Exception:
                pass
        # full-page render → provenance + input for the multimodal chart pass
        render_path = PAGE_IMG_DIR / f"{doc.doc_id}_p{i}.png"
        page.get_pixmap(dpi=CFG.render_dpi).save(render_path)
        try:
            n_drawings = len(page.get_drawings())
        except Exception:
            n_drawings = 0
        raw_text = "\n".join(t for _, t, _ in blocks)
        candidate = n_images > 0 or n_drawings >= CFG.mm_min_drawings
        pages.append(Page(doc.doc_id, i, raw_text, str(render_path),
                          n_drawings, n_images, candidate))
    pdf.close()
    return els, pages


ELEMENTS: dict[str, list[Element]] = {}
PAGES: dict[str, list[Page]] = {}
for _doc in DOCUMENTS:
    _els, _pgs = parse_document(_doc)
    ELEMENTS[_doc.doc_id] = _els
    PAGES[_doc.doc_id] = _pgs
    _cand = sum(p.is_chart_candidate for p in _pgs)
    print(f"{_doc.doc_id}: {dict(Counter(e.kind for e in _els))} · "
          f"{_cand}/{len(_pgs)} chart-candidate pages")


3c287c9e7e87: {'text': 1357, 'figure': 67, 'table': 2} · 26/28 chart-candidate pages


In [3]:
# === Stage 1.5: MULTIMODAL chart understanding (page-level vision LLM) ===
# Charts in BCG slide decks are mostly *vector* drawings, so cropping embedded
# images misses them. We instead send the full page render to the vision model
# and ask for structured JSON. Offline (Null provider) this yields nothing —
# it never fabricates numbers.
@dataclass
class ChartFact:
    doc_id: str
    page: int
    chart_id: str
    title: str | None
    subtitle: str | None
    chart_type: str | None
    x_axis: dict
    y_axis: dict
    series: list
    legend: list
    source_reference: str | None
    period: str | None
    interpretation: str
    confidence: float
    extraction_method: str  # 'multimodal_llm' | 'unavailable'
    image_path: str
    status: str  # 'extracted' | 'uncertain' | 'unavailable'

    def description(self) -> str:
        """Semantic text used for embedding + retrieval of this chart."""
        pts = []
        for s in self.series:
            name = s.get("name") or "series"
            vals = ", ".join(
                f"{v.get('period')}={v.get('value')}" for v in (s.get("values") or [])[:24]
            )
            pts.append(f"{name}: {vals}" if vals else name)
        return clean_text(
            f"CHART [{self.chart_type or 'chart'}] {self.title or ''} — {self.subtitle or ''}\n"
            f"x: {self.x_axis.get('label')} | y: {self.y_axis.get('label')} "
            f"({self.y_axis.get('unit')})\n"
            f"series: {' ; '.join(pts)}\n"
            f"interpretation: {self.interpretation}\n"
            f"source: {self.source_reference or ''} period: {self.period or ''}"
        )


def _coerce_axis(a) -> dict:
    a = a if isinstance(a, dict) else {}
    return {"label": a.get("label"), "unit": a.get("unit"), "values": a.get("values") or []}


def _grade(raw: dict) -> tuple[float, str, str]:
    """Return (confidence, extraction_method, status) with no fabrication."""
    has_numbers = any((s.get("values") for s in (raw.get("series") or [])))
    conf = float(raw.get("confidence") or (0.6 if has_numbers else 0.3))
    conf = max(0.0, min(1.0, conf))
    if not has_numbers:
        return conf, "multimodal_llm", "uncertain"
    return conf, "multimodal_llm", ("extracted" if conf >= CFG.review_confidence else "uncertain")


def extract_charts(doc: Document) -> list[ChartFact]:
    out: list[ChartFact] = []
    candidates = [p for p in PAGES[doc.doc_id] if p.is_chart_candidate]
    if CFG.mm_max_pages:
        candidates = candidates[: CFG.mm_max_pages]
    for pg in candidates:
        try:
            parsed = MM_PROVIDER.describe_page(pg.render_path, pg.raw_text)
        except Exception as exc:
            print(f"  chart_extraction failed doc={doc.doc_id} p{pg.page + 1}: {exc}")
            continue
        charts = parsed.get("charts") or []
        if not charts and not MM_PROVIDER.available:
            # provenance-preserving placeholder so the page is still reviewable
            out.append(ChartFact(
                doc.doc_id, pg.page, f"{doc.doc_id}_p{pg.page}_c0", None, None, None,
                {}, {}, [], [], None, None, "", 0.0, "unavailable", pg.render_path, "unavailable"))
            continue
        for k, c in enumerate(charts):
            conf, method, status = _grade(c)
            out.append(ChartFact(
                doc.doc_id, pg.page, f"{doc.doc_id}_p{pg.page}_c{k}",
                c.get("title"), c.get("subtitle"), c.get("chart_type"),
                _coerce_axis(c.get("x_axis")), _coerce_axis(c.get("y_axis")),
                c.get("series") or [], c.get("legend") or [],
                c.get("source"), c.get("period"),
                clean_text(str(c.get("interpretation") or "")),
                conf, method, pg.render_path, status))
    return out


CHARTS: list[ChartFact] = []
for _doc in DOCUMENTS:
    CHARTS.extend(extract_charts(_doc))

_by_status = Counter(c.status for c in CHARTS)
print(f"charts: {len(CHARTS)} · {dict(_by_status)} · provider={MM_PROVIDER.name}")
if not MM_PROVIDER.available:
    print("  (offline: set OPENAI_API_KEY to populate structured chart data — "
          "no values are invented without it)")
for _c in [c for c in CHARTS if c.status == "extracted"][:4]:
    print(f"  p{_c.page + 1} · {_c.confidence:.2f} · {(_c.title or '')[:60]}")


charts: 26 · {'unavailable': 26} · provider=null
  (offline: set OPENAI_API_KEY to populate structured chart data — no values are invented without it)


In [4]:
# === Stage 2: structured observations (indicator + period + provenance) ===
# Statistics are stored as first-class rows, NOT only inside embeddings, so they
# can be queried directly (e.g. "all CPI observations in 2025-2026").
@dataclass
class Observation:
    doc_id: str
    page: int
    indicator: str | None       # canonical macro indicator, if identifiable
    value: float
    unit: str                   # '%' | 'bps' | 'INR' | 'number'
    raw: str
    period: str | None          # economic observation period (NOT publication date)
    observation_type: str       # actual | forecast | previous | estimate | derived
    comparison: str | None      # YoY | MoM | QoQ | 'up from' | 'down from' | None
    context: str
    publication_date: str | None
    source_reference: str
    confidence: float
    extraction_method: str      # 'regex_text' | 'multimodal_chart'


_PERCENT = re.compile(r"-?\d+(?:\.\d+)?\s?%")
_BPS = re.compile(r"-?\d+(?:\.\d+)?\s?bps", re.I)
_MONEY = re.compile(
    r"(?:₹|\$|Rs\.?)\s?\d[\d,]*(?:\.\d+)?\s?"
    r"(?:lakh\s+crore|crore|lakh|billion|million|trillion|bn|mn|tn)?",
    re.I,
)
_PERIOD = re.compile(
    r"\b((?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s?'?\d{2,4}"
    r"|FY\s?\d{2,4}|Q[1-4]\s?(?:FY)?\d{0,4}|20\d{2}|H[12]\s?(?:FY)?\d{0,4})\b",
    re.I,
)
_FORECAST = re.compile(r"\b(forecast|expected|projected|estimat\w+|likely|to reach|seen at)\b", re.I)
_PREVIOUS = re.compile(r"\b(previous|prior|earlier|from|versus|vs\.?|compared)\b", re.I)
_COMPARISON = re.compile(r"\b(YoY|MoM|QoQ|year[- ]on[- ]year|up from|down from|rose to|fell to)\b", re.I)


def _to_float(s: str) -> float:
    try:
        return float(re.sub(r"[^0-9.\-]", "", s))
    except ValueError:
        return float("nan")


def _classify(ctx: str) -> str:
    if _FORECAST.search(ctx):
        return "forecast"
    if _PREVIOUS.search(ctx):
        return "previous"
    return "actual"


def extract_observations(doc: Document, page: int, text: str) -> list[Observation]:
    obs: list[Observation] = []
    for pat, unit in ((_PERCENT, "%"), (_BPS, "bps"), (_MONEY, "INR")):
        for m in pat.finditer(text):
            ctx = clean_text(text[max(0, m.start() - 110): m.end() + 110])
            per = _PERIOD.search(ctx)
            comp = _COMPARISON.search(ctx)
            obs.append(Observation(
                doc.doc_id, page, normalize_indicator(ctx), _to_float(m.group(0)), unit,
                m.group(0).strip(), per.group(1) if per else None, _classify(ctx),
                comp.group(1) if comp else None, ctx, doc.publication_date,
                f"p{page + 1}", 0.6, "regex_text"))
    return obs


def observations_from_charts(charts: list[ChartFact]) -> list[Observation]:
    """Promote legible chart data points into queryable observations (derived)."""
    out: list[Observation] = []
    for c in charts:
        if c.status != "extracted":
            continue
        ind = normalize_indicator(f"{c.title} {c.interpretation}") or None
        unit = c.y_axis.get("unit") or "number"
        for s in c.series:
            for v in (s.get("values") or []):
                val = v.get("value")
                if not isinstance(val, (int, float)):
                    continue
                out.append(Observation(
                    c.doc_id, c.page, ind, float(val), str(unit),
                    f"{val}", str(v.get("period")) if v.get("period") is not None else None,
                    "derived", None,
                    clean_text(f"{c.title or ''} {s.get('name') or ''}")[:200],
                    None, c.chart_id, c.confidence, "multimodal_chart"))
    return out


OBSERVATIONS: list[Observation] = []
for _doc in DOCUMENTS:
    for _e in ELEMENTS[_doc.doc_id]:
        if _e.kind in ("text", "table") and _e.text:
            OBSERVATIONS.extend(extract_observations(_doc, _e.page, _e.text))
OBSERVATIONS.extend(observations_from_charts(CHARTS))

_named = [o for o in OBSERVATIONS if o.indicator]
print(f"observations: {len(OBSERVATIONS)} · with-indicator={len(_named)} · "
      f"types={dict(Counter(o.observation_type for o in OBSERVATIONS))}")
for _o in _named[:8]:
    print(f"  p{_o.page + 1} [{_o.indicator}] {_o.raw!r:>10} "
          f"period={_o.period} type={_o.observation_type} ← {_o.context[:52]}")


observations: 254 · with-indicator=45 · types={'actual': 240, 'previous': 13, 'forecast': 1}
  p2 [Fiscal Deficit]     '9.9%' period=Aug'26 type=actual ← On the macro side, fiscal deficit remained the same 
  p2 [WPI]     '4.8%' period=Aug'26 type=actual ← rated in Aug'26. WPI rose to 9.9% on fuel & power an
  p2 [PMI]     '6.7%' period=FY27 type=previous ← 1, even as manufacturing PMI slipped to a 5-year low
  p2 [PMI]     '7.3%' period=FY27 type=previous ← as manufacturing PMI slipped to a 5-year low of 52.8
  p2 [GDP]     '7.8%' period=Jun'26 type=actual ← In Apr–Jun'26 (Q1 FY27), India's real GDP grew 7.8% 
  p2 [Trade Balance] '$26.9 Bn' period=None type=actual ← The merchandise trade deficit narrowed to a 5-month 
  p2 [Forex Reserves] '$17.5 Bn' period=None type=actual ← -month low of $26.9 Bn in Aug’26, on lower gold and 
  p2 [Forex Reserves]  '$741 Bn' period=None type=actual ← s. The services surplus edged down to $17.5 Bn as im


In [5]:
# === Stage 3: semantic chunking — text + tables + charts, with metadata ===
@dataclass
class Chunk:
    doc_id: str
    chunk_id: str
    page: int
    edition: str | None
    chunk_type: str             # 'text' | 'table' | 'chart'
    topic: str
    content: str
    topics: list                # canonical indicators discussed
    entities: list
    related_charts: list
    related_observations: list
    metadata: dict = field(default_factory=dict)


def _page_topic(page_els) -> str:
    texts = [e for e in page_els if e.kind == "text" and e.text]
    if not texts:
        return ""
    return max(texts, key=lambda e: e.font_max).text[:120]  # slide title = largest font


def build_chunks(doc: Document) -> list[Chunk]:
    els = ELEMENTS[doc.doc_id]
    by_page: dict[int, list] = {}
    for e in els:
        by_page.setdefault(e.page, []).append(e)
    obs_ids = {}
    for o in OBSERVATIONS:
        if o.doc_id == doc.doc_id:
            obs_ids.setdefault(o.page, []).append(o.source_reference)
    charts_by_page: dict[int, list] = {}
    for c in CHARTS:
        if c.doc_id == doc.doc_id:
            charts_by_page.setdefault(c.page, []).append(c)

    chunks: list[Chunk] = []
    for page in sorted(by_page):
        pels = by_page[page]
        topic = _page_topic(pels)
        related_charts = [c.chart_id for c in charts_by_page.get(page, [])]
        related_obs = obs_ids.get(page, [])

        # text chunks (section/page-aware, overlap)
        body = "\n".join(e.text for e in pels if e.kind == "text" and e.text)
        for ci, piece in enumerate(chunk_text(body, CFG.chunk_size, CFG.chunk_overlap)):
            content = f"[{doc.edition or doc.title} · p{page + 1}] {topic}\n{piece}"
            ents = [asdict(x) for x in extract_entities(piece)]
            topics = sorted({e["text"] for e in ents if e["label"] == "INDICATOR"})
            chunks.append(Chunk(
                doc.doc_id, f"{doc.doc_id}_p{page}_t{ci}", page, doc.edition, "text",
                topic, content, topics, ents, related_charts, related_obs,
                {"pages": [page + 1]}))

        # table chunks (textual representation + page context)
        for ti, e in enumerate(x for x in pels if x.kind == "table"):
            content = f"[{doc.edition or doc.title} · p{page + 1}] TABLE · {topic}\n{e.text}"
            chunks.append(Chunk(
                doc.doc_id, f"{doc.doc_id}_p{page}_tab{ti}", page, doc.edition, "table",
                topic, content, sorted({normalize_indicator(e.text)} - {None}),
                [], related_charts, related_obs, {"pages": [page + 1]}))

    # chart chunks (one per extracted/uncertain chart; skip pure placeholders)
    for c in CHARTS:
        if c.doc_id != doc.doc_id or c.status == "unavailable":
            continue
        topics = sorted({normalize_indicator(f"{c.title} {c.interpretation}")} - {None})
        chunks.append(Chunk(
            doc.doc_id, c.chart_id, c.page, doc.edition, "chart",
            c.title or "chart", c.description(), topics, [], [c.chart_id], [],
            {"pages": [c.page + 1], "confidence": c.confidence, "status": c.status}))
    return chunks


CHUNKS: list[Chunk] = []
for _doc in DOCUMENTS:
    CHUNKS.extend(build_chunks(_doc))

print(f"chunks: {len(CHUNKS)} · types={dict(Counter(c.chunk_type for c in CHUNKS))}")
print("top indicators discussed across the report:")
_topic_counts = Counter(t for c in CHUNKS for t in c.topics)
for _t, _n in _topic_counts.most_common(10):
    print(f"  {_n:>2}× · {_t}")


chunks: 65 · types={'text': 63, 'table': 2}
top indicators discussed across the report:
  15× · Rupee
  11× · GDP
   6× · CPI
   6× · WPI
   6× · PMI
   6× · IIP
   5× · GST
   4× · FDI
   3× · Fiscal Deficit
   3× · Trade Balance


In [6]:
# === Stage 4: embed chunks + persist local store (+ optional pgvector) ===
def _write_jsonl(path: Path, rows) -> None:
    with open(path, "w") as fh:
        for r in rows:
            fh.write(json.dumps(r, default=str) + "\n")


VECTORS = EMBEDDER.embed([c.content for c in CHUNKS])

_write_jsonl(STORE / "documents.jsonl", [asdict(d) for d in DOCUMENTS])
_write_jsonl(STORE / "observations.jsonl", [asdict(o) for o in OBSERVATIONS])
_write_jsonl(STORE / "charts.jsonl", [asdict(c) for c in CHARTS])
_write_jsonl(STORE / "chunks.jsonl",
             [{**asdict(c), "embedding": v} for c, v in zip(CHUNKS, VECTORS)])
np.save(STORE / "embeddings.npy", np.asarray(VECTORS, dtype=np.float32))
print(f"stored → {STORE.name}/: {len(DOCUMENTS)} docs · {len(CHUNKS)} chunks · "
      f"{len(OBSERVATIONS)} observations · {len(CHARTS)} charts")

# Human-review queue for low-confidence multimodal extractions (Principle 5).
REVIEW_QUEUE = [c for c in CHARTS if c.status != "unavailable"
                and c.confidence < CFG.review_confidence]
_write_jsonl(STORE / "review_queue.jsonl", [asdict(c) for c in REVIEW_QUEUE])
print(f"review queue (confidence < {CFG.review_confidence}): {len(REVIEW_QUEUE)} charts")


def write_pgvector() -> None:
    """Mirror chunks into Postgres doc_chunks(embedding vector(N)) when configured."""
    url = os.environ.get("DATABASE_URL")
    if not url:
        print("pgvector: DATABASE_URL not set — local JSONL store is the system of record")
        return
    try:
        import psycopg
    except ImportError:
        print("pgvector: psycopg not installed — skipped")
        return
    with psycopg.connect(url) as conn, conn.cursor() as cur:
        cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
        cur.execute(
            f"CREATE TABLE IF NOT EXISTS doc_chunks ("
            f"chunk_id text PRIMARY KEY, doc_id text, page int, chunk_type text, "
            f"content text, metadata jsonb, embedding vector({CFG.embed_dim}))")
        for c, v in zip(CHUNKS, VECTORS):
            cur.execute(
                "INSERT INTO doc_chunks VALUES (%s,%s,%s,%s,%s,%s,%s) "
                "ON CONFLICT (chunk_id) DO UPDATE SET embedding = EXCLUDED.embedding",
                (c.chunk_id, c.doc_id, c.page, c.chunk_type, c.content,
                 json.dumps(c.metadata), to_pgvector(v)))
        conn.commit()
    print(f"pgvector: upserted {len(CHUNKS)} chunks into doc_chunks")


write_pgvector()


stored → vector_store/: 1 docs · 65 chunks · 254 observations · 26 charts
review queue (confidence < 0.75): 0 charts
pgvector: DATABASE_URL not set — local JSONL store is the system of record


In [7]:
# === Stage 5: provenance checks + HYBRID retrieval (semantic + structured) ===
assert all(c.doc_id and c.page is not None and c.content for c in CHUNKS), "chunk missing provenance"
assert all(o.page is not None and o.source_reference for o in OBSERVATIONS), "obs missing provenance"
assert all(c.image_path for c in CHARTS), "chart missing image provenance"
print("provenance ok · chunks carry doc_id+page+type · observations carry page+period · "
      "charts carry page+image")

_M = np.asarray(VECTORS, dtype=np.float32)
_DOC_BY_ID = {d.doc_id: d for d in DOCUMENTS}


def semantic_search(query: str, k: int = 5, chunk_type: str | None = None):
    q = np.asarray(EMBEDDER.embed([query])[0], dtype=np.float32)
    sims = _M @ q  # unit-norm vectors → cosine similarity
    order = np.argsort(-sims)
    hits = []
    for i in order:
        c = CHUNKS[i]
        if chunk_type and c.chunk_type != chunk_type:
            continue
        hits.append((float(sims[i]), c))
        if len(hits) >= k:
            break
    return hits


def structured_search(indicator: str | None = None, period_contains: str | None = None,
                      obs_type: str | None = None):
    """Direct query over observations — no embeddings involved (Principle 2)."""
    res = OBSERVATIONS
    if indicator:
        canon = normalize_indicator(indicator) or indicator
        res = [o for o in res if o.indicator == canon]
    if period_contains:
        res = [o for o in res if o.period and period_contains.lower() in o.period.lower()]
    if obs_type:
        res = [o for o in res if o.observation_type == obs_type]
    return res


def hybrid_answer(query: str, indicator: str | None = None, period: str | None = None, k: int = 4):
    """Combine semantic passages + structured facts + provenance citations."""
    sem = semantic_search(query, k=k)
    facts = structured_search(indicator=indicator or normalize_indicator(query),
                              period_contains=period)
    print(f"\nQ: {query}")
    print("  — semantic evidence —")
    for score, c in sem:
        doc = _DOC_BY_ID[c.doc_id]
        print(f"    {score:.3f} · [{c.chunk_type}] p{c.page + 1} · {c.topic[:48]} "
              f"· src={doc.source[:30]}")
    if facts:
        print("  — structured observations —")
        for o in facts[:6]:
            print(f"    {o.indicator} = {o.raw} ({o.observation_type}, period={o.period}) "
                  f"· p{o.page + 1} · pub={o.publication_date}")
    return {"semantic": sem, "facts": facts}


# Semantic-only question (answer spread across narrative)
hybrid_answer("What were the main drivers of India's inflation?")
# Structured-only question (direct numeric lookup, no semantic search needed)
print("\nStructured: all CPI/WPI observations found")
for _o in structured_search(indicator="CPI")[:5] + structured_search(indicator="WPI")[:5]:
    print(f"  {_o.indicator} {_o.raw} · period={_o.period} · p{_o.page + 1} · {_o.observation_type}")
# Hybrid question (narrative + numbers together, with provenance)
hybrid_answer("How did BCG describe India's inflation trajectory and reported CPI values?",
              indicator="CPI")


provenance ok · chunks carry doc_id+page+type · observations carry page+period · charts carry page+image

Q: What were the main drivers of India's inflation?
  — semantic evidence —
    0.288 · [text] p13 · WPI inched up marginally to a record high in Aug · src=local
    0.239 · [text] p17 · India's real GDP grew by 7.8% YoY in Q1 FY27, le · src=local
    0.239 · [text] p9 · Merchandise trade deficit narrowed to a 5-month  · src=local
    0.231 · [text] p2 · Executive summary · src=local
  — structured observations —
    Fiscal Deficit = 9.9% (actual, period=Aug'26) · p2 · pub=2026-09
    WPI = 4.8% (actual, period=Aug'26) · p2 · pub=2026-09
    PMI = 6.7% (previous, period=FY27) · p2 · pub=2026-09
    PMI = 7.3% (previous, period=FY27) · p2 · pub=2026-09
    GDP = 7.8% (actual, period=Jun'26) · p2 · pub=2026-09
    None = 10% (actual, period=None) · p2 · pub=2026-09

Structured: all CPI/WPI observations found
  CPI 4.8% · period=None · p4 · actual
  WPI 4.8% · period=Aug'26 · p2 · act

{'semantic': [(0.2775169014930725,
   Chunk(doc_id='3c287c9e7e87', chunk_id='3c287c9e7e87_p12_t2', page=12, edition='Sep 2026', chunk_type='text', topic="WPI inched up marginally to a record high in Aug'26, while CPI inflation rose across most categories to reach a 20-month", content="[Sep 2026 · p13] WPI inched up marginally to a record high in Aug'26, while CPI inflation rose across most categories to reach a 20-month\nof the commodities. 3. Six categories with the highest weights in the CPI basket are shown. The CPI basket also includes excluded categories: ‘Pan, Tobacco & intoxicants’, ‘Furnishings, household equipment and routine household maintenance’, ‘Information and communication’, ‘Recreation, sport and culture’, ‘Education services’, and ‘Restaurants and accommodation services’. Source: MOSPI, CMIE, BCG Analysis\nWholesale Price Index (WPI) 1\nWPI inflation continued to inch up in Aug'26, as fuel & power and manufactured products prices climbed\nConsumer Price Index (CPI) 2\